# Liquidity and price discovery

## Summary

Does aggregate imbalance help predict price changes after accounting for event dependence?

The default workspace is **synthetic**. Its outputs establish software behavior, not empirical market findings. Run `pml demo` first, then execute this notebook top to bottom. Observed data can replace the workspace without changing the calculation functions.

## Context and methods

Fixed 10/60/300-second labels use probability points and ticks. YES assets avoid counting binary complements twice. Training precedes held-out fixtures; overlapping labels are purged. Event bootstrap intervals require at least five events.

### Key assumptions

No observed edge is claimed. Unusable books, unknown costs and unavailable labels must stay visible. Results below are derived or simulated; inspect the input provenance before interpreting them.

## Data

The immutable run manifest identifies the exact captured inputs. Set `PML_WORKSPACE` to a workspace with the same layout to analyze another capture.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import plotly.express as px

from prediction_market_lab.research import liquidity_study, load_table
from prediction_market_lab.storage import verify_run

workspace = Path(os.environ.get("PML_WORKSPACE", "data/demo")).resolve()
descriptor = json.loads((workspace / "workspace.json").read_text())
run = workspace / descriptor["run"]
manifest = verify_run(run)
features = load_table(run, "features")
provenance = sorted(features.input_provenance.unique())
print(
	{
		"input_provenance": provenance,
		"input_sha256": manifest["input_sha256"],
		"feature_rows": len(features),
	}
)
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = ["#315c8c", "#b9812b", "#78619e"]

## Results

These calculations call the same tested functions as the terminal and CLI reports.

In [ ]:
study = liquidity_study(features)
assert study["status"] == "ok"
assert set(study["split"]["train_events"]).isdisjoint(study["split"]["test_events"])
metrics = pd.DataFrame(study["forecast_metrics"])
metrics

In [ ]:
plot = metrics.dropna(subset=["mean"])
fig = px.bar(
	plot,
	x="horizon_seconds",
	y="mean",
	color="model",
	barmode="group",
	title="Held-out event-mean prediction error",
	labels={"horizon_seconds": "Horizon (seconds)", "mean": "RMSE (probability points)"},
)
fig.show()

## Validation and limitations

The following tables and checks are part of the result, including insufficient samples and negative findings.

In [ ]:
display(pd.DataFrame(study["classification"].get("metrics", [])))
display(pd.DataFrame(study["coverage"]))
print("Available labels:", study["label_availability"])
print("\n".join(study["limitations"]))

## Takeaways

Use this experiment to inspect measurement, timing and execution assumptions. For synthetic inputs, differences across models or strategies have no external validity. For observed inputs, report the covered events, exclusions, uncertainty and sensitivity before making a market claim.

Method and sources: [research protocol](../docs/research-protocol.md), [data dictionary](../docs/data-dictionary.md), [source register](../docs/sources.md).